In [ ]:
# ============================================================
# Анализ вовлечённости Telegram-канала МосУ МВД России
# Весь код в одной ячейке: запускайте её целиком (Shift+Enter).
# ============================================================
import os, ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import kruskal
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 4.5)
pd.set_option('display.max_colwidth', 90)

PATH = '/content/MVDtest.csv'
CHANNEL = 'https://t.me/mosumvdofficial/'
DAYS = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
TOPICS = {   # темы по ключевым словам, словарь можно дополнять
    'спорт': r'спорт|соревнован|турнир|чемпионат|бокс|самбо|кубок',
    'выпуск/присяга': r'выпуск|присяг|диплом',
    'наука/конференции': r'конференц|форум|научн|семинар|круглый стол',
    'праздники/память': r'праздник|поздравля|день побед|памят|ветеран',
    'абитуриенты': r'абитуриент|поступлен|приём|прием|день открытых дверей',
}

# ---------- 0. Проверка файла ----------
if not os.path.exists(PATH):
    raise FileNotFoundError(
        f'Файл {PATH} не найден. Загрузите MVDtest.csv в папку /content '
        '(значок папки слева → «Загрузить») и запустите ячейку заново.')


# ---------- 1. Загрузка и склейка сообщений в посты ----------
def join_text(s):
    return ' '.join(x for x in s if x != 'MediaMessage').strip()


def merge_reactions(s):
    total = {}
    for r in s.dropna():
        for k, v in ast.literal_eval(r).items():
            total[k] = total.get(k, 0) + v
    return total


def load_posts(path):
    df = pd.read_csv(path, sep=';')
    df['date'] = pd.to_datetime(df['date']).dt.tz_convert('Europe/Moscow')
    df = df.sort_values('message_id').reset_index(drop=True)
    # альбом = сообщения, опубликованные с разницей не больше 2 секунд
    df['post_id'] = (df['date'].diff().dt.total_seconds().abs().fillna(999) > 2).cumsum()

    posts = df.groupby('post_id').agg(
        date=('date', 'min'),
        message_id=('message_id', 'min'),
        n_items=('message_id', 'count'),
        text=('message', join_text),
        views=('views', 'max'),               # просмотры альбома ≈ максимум по элементам
        reactions=('reactions_total', 'sum'),
        forwards=('forwards', 'max'),
        comments=('comments', lambda s: s.fillna(0).sum()),
        forwarded=('forwarded', 'max'),
        reactions_dict=('reactions', merge_reactions),
    ).reset_index(drop=True)

    members = df['channel_members_count'].iloc[0]

    # метрики
    posts['interactions'] = posts['reactions'] + posts['forwards'] + posts['comments']
    posts['ER'] = posts['interactions'] / posts['views'] * 100
    posts['ERR'] = posts['views'] / members * 100

    # признаки поста — все создаются здесь, чтобы ниже ничего не потерялось
    posts['text_len'] = posts['text'].str.len()
    posts['log_len'] = np.log1p(posts['text_len'])
    posts['format'] = np.select(
        [posts['n_items'] > 1, posts['text_len'] > 0],
        ['альбом', 'одно медиа/текст'], default='медиа без подписи')
    posts['month'] = posts['date'].dt.tz_localize(None).dt.to_period('M')
    posts['weekday'] = posts['date'].dt.dayofweek.map(dict(enumerate(DAYS)))
    posts['hour'] = posts['date'].dt.hour
    posts['afternoon'] = (posts['hour'] >= 15).astype(int)
    posts['hashtags'] = posts['text'].str.lower().str.findall(r'#\w+')
    posts['n_tags'] = posts['hashtags'].str.len()
    for t, pattern in TOPICS.items():
        posts[t] = posts['text'].str.lower().str.contains(pattern, regex=True)
    posts['date_str'] = posts['date'].dt.strftime('%d.%m.%Y %H:%M')
    posts['link'] = CHANNEL + posts['message_id'].astype(str)
    return df, posts, members


df, posts, members = load_posts(PATH)
print(f'Строк в файле: {len(df)}, постов после склейки: {len(posts)}')
print(f'Подписчиков: {members}')
print(f'Период: {posts.date.min():%d.%m.%Y} – {posts.date.max():%d.%m.%Y}')

METRICS = ['views', 'reactions', 'forwards', 'comments', 'ER', 'ERR']


# ---------- 2. Общие показатели ----------
print('\n=== 2. Общие показатели ===')
print(f'Постов хотя бы с одной реакцией: {(posts.reactions > 0).mean():.0%}')
print('ERR может быть выше 100%: в файле только текущее число подписчиков, '
      'а просмотры набирают и люди без подписки.')
display(pd.DataFrame({
    'медиана': posts[METRICS].median(),
    'среднее': posts[METRICS].mean(),
    'максимум': posts[METRICS].max(),
}).round(2))


# ---------- 3. Динамика по месяцам ----------
print('\n=== 3. Динамика по месяцам ===')
monthly = posts.groupby('month').agg(
    posts=('views', 'size'),
    views_median=('views', 'median'),
    ER_median=('ER', 'median'),
    ERR_median=('ERR', 'median'),
)
display(monthly.round(2))

fig, ax1 = plt.subplots()
x = monthly.index.astype(str)
ax1.bar(x, monthly['posts'], color='#c9d6e8', label='постов')
ax1.set_ylabel('постов в месяц')
ax1.tick_params(axis='x', rotation=45)
ax2 = ax1.twinx()
ax2.plot(x, monthly['ER_median'], color='#d1495b', marker='o', label='медианный ER, %')
ax2.set_ylabel('ER, %')
ax2.grid(False)
fig.legend(loc='upper right')
plt.title('Число постов и вовлечённость по месяцам')
plt.tight_layout(); plt.show()


# ---------- 4. День недели и час ----------
print('\n=== 4. День недели и час публикации ===')
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=posts, x='weekday', y='ER', order=DAYS, ax=axes[0],
            showfliers=False, color='#9cc5a1')
axes[0].set_title('ER по дням недели')
sns.boxplot(data=posts, x='hour', y='ER', ax=axes[1], showfliers=False, color='#9cc5a1')
axes[1].set_title('ER по часам (МСК)')
plt.tight_layout(); plt.show()

pivot = posts.pivot_table(index='weekday', columns='hour', values='ER', aggfunc='median').reindex(DAYS)
counts = posts.pivot_table(index='weekday', columns='hour', values='ER', aggfunc='size').reindex(DAYS)
pivot = pivot.where(counts >= 3)   # ячейки, где меньше 3 постов, скрыты как ненадёжные
plt.figure(figsize=(13, 3.8))
sns.heatmap(pivot, cmap='YlGn', annot=True, fmt='.1f', cbar_kws={'label': 'медианный ER, %'})
plt.title('Медианный ER: день × час (только ячейки с ≥3 постами)')
plt.show()

display(posts.groupby('weekday')['ER'].agg(['size', 'median']).reindex(DAYS).round(2))
stat, p = kruskal(*[g['ER'].values for _, g in posts.groupby('weekday')])
print(f'Kruskal-Wallis по дням недели: H={stat:.2f}, p={p:.4f} → '
      + ('разница значима' if p < 0.05 else 'значимой разницы нет'))


# ---------- 5. Формат и длина текста ----------
print('\n=== 5. Формат поста и длина текста ===')
display(posts.groupby('format').agg(
    posts=('views', 'size'),
    views_median=('views', 'median'),
    ER_median=('ER', 'median'),
    reactions_median=('reactions', 'median'),
    forwards_median=('forwards', 'median'),
).round(2))

display(posts.groupby(posts['forwarded'].map({0: 'свой', 1: 'пересланный'}))[['views', 'ER']]
        .agg(['median', 'size']).round(2))

with_text = posts[posts.text_len > 0].copy()
with_text['len_group'] = pd.cut(with_text['text_len'], [0, 300, 600, 1000, 10_000],
                                labels=['до 300', '300–600', '600–1000', '1000+'])
display(with_text.groupby('len_group', observed=True)[['views', 'ER']]
        .agg(['median', 'size']).round(2))

print('Корреляция Спирмена:')
display(posts[['text_len', 'n_items', 'views', 'reactions', 'forwards', 'ER']]
        .corr(method='spearman').round(2))


# ---------- 6. Хэштеги и темы ----------
print('\n=== 6. Хэштеги и темы ===')
tags = posts.explode('hashtags').dropna(subset=['hashtags'])
tag_stats = tags.groupby('hashtags').agg(posts=('ER', 'size'), ER_median=('ER', 'median'),
                                         views_median=('views', 'median'))
display(tag_stats[tag_stats.posts >= 5].sort_values('ER_median', ascending=False).round(2).head(15))

topic_rows = [{'тема': t, 'постов': int(posts[t].sum()),
               'views_median': posts.loc[posts[t], 'views'].median(),
               'ER_median': posts.loc[posts[t], 'ER'].median()} for t in TOPICS]
display(pd.DataFrame(topic_rows).sort_values('ER_median', ascending=False).round(2))


# ---------- 7. Реакции ----------
print('\n=== 7. Какие реакции ставят ===')
all_r = {}
for d in posts['reactions_dict']:
    for k, v in d.items():
        all_r[k] = all_r.get(k, 0) + v
react = pd.Series(all_r, dtype='int64').sort_values(ascending=False)
display(pd.DataFrame({'кол-во': react, 'доля, %': (react / react.sum() * 100).round(1)}).head(10))
react.head(10).sort_values().plot.barh(color='#edae49')
plt.title('Топ-10 реакций'); plt.xlabel('количество'); plt.show()


# ---------- 8. Лучшие и худшие посты ----------
print('\n=== 8. Лучшие и худшие посты ===')
cols = ['date_str', 'views', 'reactions', 'forwards', 'ER', 'format', 'text', 'link']
print('ТОП-10 по ER (среди постов с просмотрами не ниже медианы):')
display(posts[posts.views >= posts.views.median()].nlargest(10, 'ER')[cols].round(2))
print('ТОП-10 по просмотрам:')
display(posts.nlargest(10, 'views')[cols].round(2))
print('10 постов с самым низким ER:')
display(posts.nsmallest(10, 'ER')[cols].round(2))


# ---------- 9. Регрессия ----------
print('\n=== 9. Что связано с ER (регрессия, не доказывает причинность) ===')
model = smf.ols(
    "ER ~ log_len + n_items + n_tags + afternoon + C(weekday, Treatment(reference='Пн'))",
    data=posts).fit()
print(model.summary().tables[1])
print(f'R² = {model.rsquared:.3f}')


# ---------- 10. Сохранение ----------
posts.drop(columns=['reactions_dict']).to_csv('/content/posts_engagement.csv',
                                              index=False, encoding='utf-8-sig')
monthly.to_csv('/content/monthly_engagement.csv', encoding='utf-8-sig')
print('\nСохранено: /content/posts_engagement.csv, /content/monthly_engagement.csv')
# чтобы скачать: from google.colab import files; files.download('/content/posts_engagement.csv')
